In [1]:
import pandas as pd
from Bio import SeqIO

In [2]:
# Files for Saccharomyces cerevisiae
anno_sc = "results/saccharomices_cerevisiae/saccharomyces_cerevisiae.gff"
sample_id_sc = "saccharomyces_cerevisiae"
filtered_gff_sc = f"{sample_id_sc}.filtered.gff"
gffread_output_sc = "results/saccharomices_cerevisiae/gffread/saccharomyces_cerevisiae.faa"
ref_sc = "results/saccharomices_cerevisiae/saccharomyces_cerevisiae.tsv"


# Files for Drosophila melanogaster
anno_dm = "results/drosophila_melanogaster/reference/genomic.gff"
sample_id_dm = "drosophila_melanogaster"
filtered_gff_dm = f"{sample_id_dm}.filtered.gff"
gffread_output_dm = "results/drosophila_melanogaster/gffread/drosophila_melanogaster_gffread.faa"
ref_dm = "drosophila_melanogaster.tsv"

In [3]:
%%bash -s "$anno_sc" "$sample_id_sc" "$filtered_gff_sc"

anno_sc="$1"
sample_id_sc="$2"
filtered_gff_sc="$3"

set -euo pipefail

awk 'BEGIN{FS=OFS="\t"}
       /^#/ {print; next}
       $7=="?" {next}
       $9 ~ /exception=trans-splicing/ {next}
       {print}' "${anno_sc}" > "${sample_id_sc}.filtered.gff"

total=$(grep -vc '^#' "${anno_sc}" || true)
kept=$(grep -vc '^#' "${filtered_gff_sc}" || true)
removed=$(( total - kept ))

echo "[GFFREAD/${sample_id_sc}] total=$total kept=$kept removed=$removed"

[GFFREAD/saccharomyces_cerevisiae] total=27135 kept=27135 removed=0


In [4]:
# Remove the raw with #
gff_df = pd.read_csv(filtered_gff_sc, sep="\t", comment="#", header=None)
# remove columns 0, 1, 3, 4, 5, 6, 7
gff_df = gff_df.drop(columns=[0, 1, 3, 4, 5, 6, 7])
# On 2 columns remove all rows exception for mRNA
gff_df = gff_df[gff_df[2].str.contains("mRNA", na=False)]
# remove on 8 informations before firts ;
gff_df[8] = gff_df[8].str.split(";").str[0]
# on 8 remove the term "ID="
gff_df[8] = gff_df[8].str.replace("ID=", "", regex=False)
gff_df_final = gff_df.drop(columns=[2])
gff_df_final.to_csv(f"{sample_id_sc}.filtered.tsv", sep="\t", index=False, header=False)

gff_df

,2,8
4,mRNA,rna-NM_001180043.1
8,mRNA,rna-NM_001184582.1
12,mRNA,rna-NM_001178208.1
17,mRNA,rna-NM_001179897.1
21,mRNA,rna-NM_001180042.1
...,...,...
27040,mRNA,rna-Q0140
27052,mRNA,rna-Q0160
27104,mRNA,rna-Q0250
27108,mRNA,rna-Q0255


In [5]:
# Now we need prepared the output of gffread to be able to compare it with the reference
records = []

for record in SeqIO.parse(gffread_output_sc, "fasta"): # read the fasta file generated by gffread
    header = record.id # get the id of the fasta file
    records.append(header) # append the id to the list of records
records

gffread_df = pd.DataFrame(records, columns=["ID"])

# Save as a tsv file
gffread_df.to_csv(f"{sample_id_sc}_gffread.tsv", sep="\t", index=False, header=True)

gffread_df


,ID
0,rna-NM_001180043.1
1,rna-NM_001184582.1
2,rna-NM_001178208.1
3,rna-NM_001179897.1
4,rna-NM_001180042.1
...,...
5999,rna-NM_001184299.1
6000,rna-NM_001184300.1
6001,rna-NM_001184301.1
6002,rna-Q0080


In [6]:
# ref
ref_df = pd.read_csv(ref_sc, sep="\t", comment="#", header=None)
# Comparated the column 0 of gff_df_final with column 0 of ref_df and keep only the row that are only in ref_df
only_ref_df = ref_df[~ref_df[0].isin(gffread_df["ID"])]

only_ref_df

,0,1
170,rna-Q0045,cytochrome c oxidase subunit 1 (mitochondrion)
171,rna-Q0070,intron-encoded DNA endonuclease aI5 alpha (mit...
172,rna-Q0065,intron-encoded DNA endonuclease aI4 (mitochond...
173,rna-Q0060,intron-encoded DNA endonuclease aI3 (mitochond...
174,rna-Q0055,intron-encoded reverse transcriptase aI2 (mito...
175,rna-Q0050,intron-encoded reverse transcriptase aI1 (mito...
176,rna-Q0075,intron-encoded DNA endonuclease aI5 beta (mito...
178,rna-Q0085,F1F0 ATP synthase subunit a (mitochondrion)
179,rna-Q0105,cytochrome b (mitochondrion)
180,rna-Q0120,intron-encoded RNA maturase bI4 (mitochondrion)


In [7]:
%%bash -s "$anno_dm" "$sample_id_dm" "$filtered_gff_dm"

anno_dm="$1"
sample_id_dm="$2"
filtered_gff_dm="$3"

set -euo pipefail

awk 'BEGIN{FS=OFS="\t"}
       /^#/ {print; next}
       $7=="?" {next}
       $9 ~ /exception=trans-splicing/ {next}
       {print}' "${anno_dm}" > "${sample_id_dm}.filtered.gff"

total=$(grep -vc '^#' "${anno_dm}" || true)
kept=$(grep -vc '^#' "${filtered_gff_dm}" || true)
removed=$(( total - kept ))

echo "[GFFREAD/${sample_id_dm}] total=$total kept=$kept removed=$removed"

[GFFREAD/drosophila_melanogaster] total=414876 kept=414627 removed=249


In [8]:
# Remove the raw with #
gff_df_dm = pd.read_csv(filtered_gff_dm, sep="\t", comment="#", header=None)
# remove columns 0, 1, 3, 4, 5, 6, 7
gff_df_dm = gff_df_dm.drop(columns=[0, 1, 3, 4, 5, 6, 7])
# On 2 columns remove all rows exception for mRNA
gff_df_dm = gff_df_dm[gff_df_dm[2].str.contains("mRNA", na=False)]
# remove on 8 informations before firts ;
gff_df_dm[8] = gff_df_dm[8].str.split(";").str[0]
# on 8 remove the term "ID="
gff_df_dm[8] = gff_df_dm[8].str.replace("ID=", "", regex=False)
gff_df_final = gff_df_dm.drop(columns=[2])
gff_df_final.to_csv(f"{sample_id_dm}.filtered.tsv", sep="\t", index=False, header=False)

gff_df_dm

,2,8
16,mRNA,rna-NM_001103384.3
23,mRNA,rna-NM_001258513.2
31,mRNA,rna-NM_001258512.2
39,mRNA,rna-NM_001297796.1
67,mRNA,rna-NM_001297795.1
...,...,...
414586,mRNA,rna-Dmel_CG34085
414590,mRNA,rna-Dmel_CG34086
414600,mRNA,rna-Dmel_CG34089
414604,mRNA,rna-Dmel_CG34090


In [14]:
# Now we need prepared the output of gffread to be able to compare it with the reference
records_dm = []

for record in SeqIO.parse(gffread_output_dm, "fasta"): # read the fasta file generated by gffread
    header_dm = record.id # get the id of the fasta file
    records_dm.append(header_dm) # append the id to the list of records
records_dm

gffread_df_dm = pd.DataFrame(records_dm, columns=["ID"]) # create a dataframe with the list of records and the column name "ID"

# Save as a tsv file
gffread_df_dm.to_csv(f"{sample_id_dm}_gffread.tsv", sep="\t", index=False, header=True) # save the dataframe as a tsv file with the name of the sample and the suffix "_gffread.tsv"

record

SeqRecord(seq=Seq('MSRNQNESNLLDINEDLPFALALPPVSMDSLQLNADGNSPEDKTLGIGRKTILN...RRT'), id='rna-NM_001015100.2', name='rna-NM_001015100.2', description='rna-NM_001015100.2 Dbxref=FLYBASE:FBtr0113685,GeneID:3354863,GenBank:NM_001015100.2,FLYBASE:FBgn0040038;Name=NM_001015100.2;Note=klhl10-RA%3B Dmel\\klhl10-RA%3B CG12423-RA%3B Dmel\\CG12423-RA;gbkey=mRNA;gene=klhl10;locus_tag=Dmel_CG12423;orig_protein_id=gnl|FlyBase|CG12423-PA|gb|EAA46325;orig_transcript_id=gnl|FlyBase|CG12423-RA;product=kelch like family member 10%2C transcript variant A;transcript_id=NM_001015100.2;description=kelch like family member 10;gene_biotype=protein_coding;gene_synonym=anon-WO0140519.152,CG12423,Dmel\\CG12423,Klhl10;CDS_Dbxref=FLYBASE:FBpp0112408,GeneID:3354863,GenBank:NP_001015100.2,FLYBASE:FBgn0040038;CDS_Name=NP_001015100.2;CDS_gbkey=CDS;CDS_product=kelch like family member 10%2C isoform A;protein_id=NP_001015100.2', dbxrefs=[])

In [10]:
# ref
ref_df_dm = pd.read_csv(ref_dm, sep="\t", comment="#", header=None)
# Comparated the column 0 of gff_df_final with column 0 of ref_df and keep only the row that are only in ref_df
only_ref_df_dm = ref_df_dm[~ref_df_dm[0].isin(gffread_df_dm["ID"])]

only_ref_df_dm

,0,1
5,rna-NM_001014456.1,gustatory receptor 22b
35,rna-NM_001014492.2,uncharacterized protein Dmel_CG33511
60,rna-NM_001014520.3,"pipsqueak, isoform M"
162,rna-NM_001014631.3,"couch potato, isoform T"
163,rna-NM_001014632.4,"couch potato, isoform S"
...,...,...
30797,rna-Dmel_CG34085,NADH dehydrogenase subunit 4 (mitochondrion)
30798,rna-Dmel_CG34086,NADH dehydrogenase subunit 4L (mitochondrion)
30799,rna-Dmel_CG34089,NADH dehydrogenase subunit 6 (mitochondrion)
30800,rna-Dmel_CG34090,cytochrome b (mitochondrion)
